# 18 günlük ayrışma testi — ECG-FM gömmeleri (ikinci EKG modeli)

**1)** **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU → Kaydet**

**2)** **Çalışma zamanı → Tümünü çalıştır** (uyarı çıkarsa *Yine de çalıştır*), Drive iznini ver.

Yalnızca gömme çıkarır (eğitim yok): 1136 pencere × 6 model (ECG-FM ön eğitimli + 5 rastgele başlangıç).
İlk çalıştırmada ECG-FM ağırlıkları (~1 GB) indirilir. Her modelin süresi ekrana yazılır.
Sonuçlar Drive'a (`bitkiEkg_sonuclar/18gun/`) kaydedilir; oturum koparsa **Tümünü çalıştır** yeter, bitenler atlanır.
Yöntem önceden yazıldı: `docs/on_kayit.md` Ek 1. Bitince bana **"bitti"** yaz.

## 1. GPU kontrolü

In [ ]:
import torch
assert torch.cuda.is_available(), 'GPU yok! Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU, sonra Tümünü çalıştır.'
print('GPU:', torch.cuda.get_device_name(0), '| torch', torch.__version__)

## 2. Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
KOK = Path('/content/drive/MyDrive/bitkiEkg_sonuclar/18gun')
KOK.mkdir(parents=True, exist_ok=True)
print('Sonuç klasörü:', KOK)

## 3. Kod, kütüphaneler ve veri

In [ ]:
!rm -rf /content/bitkiEkg /content/fairseq-signals
!git clone -q -b onDeneme --depth 1 https://github.com/melihakcam/bitkiEkg.git /content/bitkiEkg
!git clone -q https://github.com/Jwoo5/fairseq-signals.git /content/fairseq-signals
!cd /content/fairseq-signals && git checkout -q f8f0ff1c788a82c2059cb452cd5462898867489e
!pip install -q cffi hydra-core omegaconf regex "sacrebleu>=1.4.12" wfdb "transformers>=4.40,<5" huggingface_hub
import sys
sys.path.insert(0, '/content/bitkiEkg/src')
sys.path.insert(0, '/content/fairseq-signals')
import time
import numpy as np
from bitki_ekg.ekg_girdi import ecgfm_girdisi
from bitki_ekg.ince_ayar import ecgfm_yukle
from bitki_ekg.sonda import ecgfm_gomme

z = np.load('/content/bitkiEkg/data/colab/domates_18gun_6h_500.npz')
X = ecgfm_girdisi(z['L'].astype(np.float32))
assert X.shape == (1136, 12, 2500), X.shape
print('Veri:', X.shape, '| bitki:', len(np.unique(z['bitki'])), '| gün:', len(np.unique(z['gun'])))

## 4. Gömmeler: ECG-FM ağırlıkları + 5 farklı rastgele başlangıç

In [ ]:
KOLLAR = [('ecgfm', True, 42)] + [(f'ecgfm_rastgele_t{t}', False, t) for t in [1, 2, 3, 4, 5]]
t_bas = time.time()
for i, (kol, onceden, tohum) in enumerate(KOLLAR, 1):
    dosya = KOK / f'gomme_{kol}_18gun.npz'
    if dosya.exists():
        print(f'[{i}/6] {kol}: önceden tamamlanmış, atlandı'); continue
    t0 = time.time()
    govde = ecgfm_yukle(onceden, tohum)
    E = ecgfm_gomme(govde, X, 'cuda')
    assert np.isfinite(E).all(), f'{kol}: gömmede NaN/sonsuz değer'
    np.savez_compressed(dosya, E=E, bitki=z['bitki'], gun=z['gun'], saat=z['saat'])
    del govde; torch.cuda.empty_cache()
    print(f'[{i}/6] {kol}: {E.shape}, {time.time() - t0:.0f} s | toplam {(time.time() - t_bas) / 60:.1f} dk', flush=True)
print('Bitti. Dosyalar:', sorted(p.name for p in KOK.glob('gomme_ecgfm*_18gun.npz')))